# PianoVAM video -> MIDI: un esempio del dataset, con il MIDI stimato

Questo notebook prende **un esempio qualsiasi del dataset** (lo scegli con un indice, da 1 a
107), mostra il **video con l'audio originale**, il **MIDI registrato dal pianoforte
digitale** e il **MIDI stimato dal modello** a partire dal solo video, e mette i due MIDI
uno accanto all'altro: metriche, piano roll e ascolto A/B.

Se invece vuoi provare un **video tuo** metti `MODE = "upload"`: non c'e' nessun MIDI di
riferimento da confrontare, ma il modello restituisce comunque un MIDI stimato.

Dataset: [`PianoVAM/PianoVAM_v1`](https://huggingface.co/datasets/PianoVAM/PianoVAM_v1)
(107 registrazioni, licenza **CC BY-NC-SA 4.0**). Il modello e' quello del progetto
`pianovam-video2midi`: dalla striscia della tastiera rettificata in 88 colonne (una per
tasto) stima quattro mappe di probabilita' (onset, tasto premuto, rilascio, velocity) e da
li' ricava le note.

## Cosa fa, cella per cella

| Cella | Contenuto |
|---|---|
| 0 | configurazione: modo, indice dell'esempio, secondi da elaborare, soglia |
| 1 | codice e pesi del modello (li prende da soli; se serve, chiede uno zip) |
| 2 | **l'indice**: la tabella degli esempi (1..107) e la scheda di quello scelto |
| 3 | **video + audio originale** (estratto da guardare in Colab; il file intero resta sul disco) |
| 4 | il **sintetizzatore** condiviso e il **MIDI originale** (note, statistiche, ascolto) |
| 5 | **la stima**: rettifica della tastiera, cache, modello, note, MIDI |
| 6 | **il confronto**: P/R/F1, errore di onset, ascolto A/B dei due MIDI |
| 6b | **piano roll**: il MIDI originale e quello stimato, uno sopra l'altro |
| 7 | **video tuo** (`MODE = "upload"`): carichi un mp4, leggi i 4 angoli della tastiera, stima |

## Tempi e costi, in breve

- il video di una registrazione pesa ~340 MB e arriva dal Hub quando serve (cella 3);
- l'elaborazione si ferma ai primi `SECONDS` secondi (60 di default): cosi' una decina di
  secondi di decodifica e meno di un minuto di modello. Con `SECONDS = None` si elabora
  tutta la registrazione (minuti, non secondi);
- il MIDI stimato lo puoi scaricare e riascoltare anche fuori da Colab.

## Cosa aspettarsi (numeri veri, non promesse)

Il modello e' piccolo (~0,3 M di parametri) e onesto va detto che la qualita' dipende da
quanto il pianista del video somiglia a quelli visti in addestramento. F1 sulle note a
tolleranza 50 ms, soglia 0.70, checkpoint m6 epoca 9:

| Registrazione | Pianista | F1 | Precision | Recall |
|---|---|---|---|---|
| `2024-02-17_21-44-37` | Changyun (test ufficiale) | **0.73** | 0.79 | 0.68 |
| `2024-09-05_20-46-25` | Yujeong (mai visto) | **0.93** | 0.96 | 0.90 |
| `2024-09-05_21-13-49` | Yujeong (mai visto) | **0.94** | 0.96 | 0.92 |
| `2024-02-20_18-08-35` | Taegyun (mai visto) | **0.82** | 0.80 | 0.85 |
| `2024-02-22_11-58-09` | Taegyun (mai visto) | **0.80** | 0.81 | 0.79 |
| `2024-09-04_19-52-57` | Minwook (mai visto) | **0.47** | 0.87 | 0.32 |
| `2024-09-04_21-09-37` | Minwook (mai visto) | **0.48** | 0.88 | 0.33 |

Su un pianista nuovo il modello tende a **non accorgersi** delle note (recall basso) invece
di suonare a caso: dove sbaglia, quasi sempre tace. Il caso peggiore e' la musica densa e
veloce (Minwook): li' il recall e' basso e non si recupera abbassando la soglia, perche'
il modello non *vede* quelle note.

Sul video tuo vale la stessa regola: se il pianista, l'inquadratura o la tastiera somigliano
a quelli del dataset (vista dall'alto, tastiera intera in campo, 60 fps) funziona bene;
altrimenti aspettati numeri come quelli di Minwook.

In [ ]:
# ======================================================================
# 0) CONFIGURAZIONE
# ======================================================================
# Due modi d'uso:
#   MODE = "dataset" -> scegli un esempio con il suo indice (1..107) e confronti
#                       il MIDI stimato con quello registrato dal pianoforte;
#   MODE = "upload"  -> usi un video tuo: nessun MIDI di riferimento, ma il
#                       MIDI stimato arriva lo stesso (cella 7).
MODE = "dataset"

from pathlib import Path
from glob import glob
import datetime
import importlib
import json
import re
import subprocess
import sys
import tempfile
import zipfile

import numpy as np
import pandas as pd

# --- il modello (codice + pesi) ---------------------------------------
CODE_DIR = "/content/pianovam-video2midi"   # dove stanno src/ e runs/ (cella 1)
GITHUB_REPO = "https://github.com/Alessandro1040/pianovam-video2midi.git"
BUNDLE_URL = None            # zip con dentro src/ + pesi, se non usi git (opzionale)
CHECKPOINT = None            # None = runs/m6_2ch/last.pt (m6, epoca 9)

# --- modo "dataset" ---------------------------------------------------
DATASET_ROOT = "PianoVAM_v1.2"   # cartella con metadata.json, Video/, MIDI/, TSV/
RECORD_INDEX = None          # 1..107 (tabella della cella 2); None = usa RECORD_TIME
RECORD_TIME = "2024-02-17_21-44-37"   # nome della registrazione (prevale sull'indice)

# --- modo "upload" ----------------------------------------------------
UPLOAD_VIDEO = None          # percorso del tuo mp4 (in Colab lo carichi nella cella 7)
QUAD = None                  # "x,y x,y x,y x,y" LT RT RB LB, letto dalla griglia

# --- quanta registrazione elaborare, e come ---------------------------
SECONDS = 60                 # primi N secondi (None = tutta la registrazione: lento)
THRESHOLD = 0.70             # soglia sugli onset (0.70 = modello m6 epoca 9)
DEVICE = "auto"              # "auto" (usa la GPU di Colab se c'e'), "cpu", "cuda"
AUTO_DOWNLOAD = True         # scarica dal Hub i file che mancano
SHOW_VIDEO = True            # mostra l'estratto del video (il file intero e' ~340 MB)
VIDEO_EXCERPT_SECONDS = 15   # secondi di video da mostrare, a 640 px
OUT_DIR = "video2midi_out"   # cache, MIDI stimati e figure finiscono qui

# Colab: se hai il dataset su Google Drive, monta il Drive e cambia DATASET_ROOT:
#   from google.colab import drive
#   drive.mount("/content/drive")
#   DATASET_ROOT = "/content/drive/MyDrive/PianoVAM_v1.2"
REPO_HUB = "PianoVAM/PianoVAM_v1"    # dataset su Hugging Face
CACHE_ROOT = None            # None = OUT_DIR/cache (o la cache che trovo accanto al dataset)

# --- dettagli del modello e dell'ascolto ------------------------------
TOP_MARGIN = 0.4             # margine incluso sopra la tastiera, quando la ritaglio
WINDOW = 256                 # frame per finestra di inferenza (stride < window: i
STRIDE = 192                 # bordi di finestra sono il punto debole, si mediano)
SAMPLE_RATE = 22050          # sintetizzatore: MIDI originale e MIDI stimato, stesso suono

In [ ]:
# --- funzioni di servizio, usate da tutte le celle ---------------------
def titolo(testo):
    """Intestazione di cella, per orientarsi nell'output."""
    print()
    print("=" * 78)
    print(testo)
    print("=" * 78)


def human(size_bytes):
    """Dimensione leggibile."""
    valore = float(size_bytes)
    for unita in ["B", "KB", "MB", "GB"]:
        if valore < 1024:
            return "%.1f %s" % (valore, unita)
        valore /= 1024
    return "%.1f TB" % valore


def durata_breve(valore):
    """'0 days 00:12:25.493000' -> '12:25' (la metadata.json scrive cosi')."""
    try:
        totale = int(round(pd.Timedelta(valore).total_seconds()))
        return "%d:%02d" % (totale // 60, totale % 60)
    except Exception:
        return str(valore)


def find_dataset_root(configurato):
    """Cartella del dataset: quella che contiene metadata.json."""
    candidati = [Path(configurato),
                 Path.cwd() / "PianoVAM_v1.2",
                 Path.cwd() / "PianoVAM_v1",
                 Path("/content/PianoVAM_v1.2"),
                 Path("/content/drive/MyDrive/PianoVAM_v1.2"),
                 Path.home() / "PianoVAM_v1.2"]
    cache = Path.home() / ".cache/huggingface/hub/datasets--PianoVAM--PianoVAM_v1/snapshots"
    candidati += [Path(p) for p in glob(str(cache / "*"))]
    for candidato in candidati:
        if (candidato / "metadata.json").is_file():
            return candidato.resolve()
    raise FileNotFoundError(
        "metadata.json non trovato: ho cercato in\n"
        + "\n".join("  - %s" % p for p in candidati[:7]) + "\n"
        "Metti in cima al notebook DATASET_ROOT = la cartella del dataset (quella con\n"
        "metadata.json e le sottocartelle Video/ MIDI/ TSV/), oppure lascia\n"
        "AUTO_DOWNLOAD = True e la cella 2 se lo scarica dal Hub (66 KB).")


def scarica_dal_hub(root, relativo):
    """Scarica un file del dataset (metadata.json, o una modalita' di una registrazione)."""
    try:
        from huggingface_hub import hf_hub_download
    except ImportError:
        print("Serve huggingface_hub: esegui  !pip install -q huggingface_hub  e riprova.")
        return None
    if not AUTO_DOWNLOAD:
        print("%s non e' sul disco e AUTO_DOWNLOAD = False." % relativo)
        print("Metti AUTO_DOWNLOAD = True nella cella 0 e riesegui questa cella,")
        print("oppure copia il file a mano in %s" % (Path(root) / relativo))
        return None
    print("Scarico %s dal Hub in %s ..." % (relativo, root))
    try:
        hf_hub_download(REPO_HUB, relativo, repo_type="dataset", local_dir=str(root))
    except Exception as errore:
        print("Download non riuscito (%s): %s" % (type(errore).__name__, errore))
        return None
    percorso = Path(root) / relativo
    if not percorso.is_file():
        print("Il Hub non ha %s: questa registrazione non ha quella modalita'." % relativo)
        return None
    print("Fatto:", human(percorso.stat().st_size))
    return percorso


def file_dataset(root, relativo):
    """Path di un file del dataset, scaricandolo se manca (None se non c'e' e non si puo')."""
    percorso = Path(root) / relativo
    if percorso.is_file():
        return percorso
    return scarica_dal_hub(root, relativo)

In [ ]:
def mostra_tabella(tabella, righe=12):
    """Tabella in Colab; fuori da Colab (o senza display) stampa le prime righe."""
    try:
        from IPython.display import display

        display(tabella)
    except Exception:
        print(tabella.head(righe).to_string(index=False))
        if len(tabella) > righe:
            print("(e altre %d righe)" % (len(tabella) - righe))


def riproduci_audio(campioni, sample_rate, etichetta):
    """Riproduce l'audio in Colab; se IPython non c'e' salva un WAV e stampa il percorso."""
    campioni = np.asarray(campioni)
    try:
        from IPython.display import Audio, display

        return display(Audio(data=campioni, rate=sample_rate, normalize=False))
    except Exception:
        import wave as _wave

        percorso = Path(OUT_DIR) / (etichetta + ".wav")
        percorso.parent.mkdir(parents=True, exist_ok=True)
        with _wave.open(str(percorso), "wb") as handle:
            handle.setnchannels(1)
            handle.setsampwidth(2)
            handle.setframerate(int(sample_rate))
            handle.writeframes((np.clip(campioni, -1, 1) * 32767).astype("<i2").tobytes())
        print("(audio salvato in %s: fuori da Colab non c'e' il lettore)" % percorso)
        return None


def cartella_modello():
    """La cartella del progetto (con dentro src/ e runs/) se ce n'e' gia' una a portata."""
    for candidata in (Path(CODE_DIR), Path.cwd(), Path.cwd() / "pianovam-video2midi"):
        if (candidata / "src").is_dir() and (candidata / "runs").is_dir():
            return candidata
    return None


def scarica_o_chiedi_modello(destinazione):
    """Porta in `destinazione` il codice (src/) e i pesi del modello.

    Tre strade, in ordine: il progetto c'e' gia' (nella cartella di lavoro o in CODE_DIR);
    `git clone` del repo pubblico; uno zip (URL in BUNDLE_URL oppure caricato a mano con
    files.upload, per esempio quello di `python3 scripts/pack_colab_bundle.py`).
    """
    destinazione = Path(destinazione)
    trovata = cartella_modello()
    if trovata is not None:
        print("Codice gia' presente in %s" % trovata)
        return trovata

    if GITHUB_REPO:
        print("Provo a clonare %s ..." % GITHUB_REPO)
        risultato = subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO, str(destinazione)],
                                   capture_output=True, text=True)
        if risultato.returncode == 0 and (destinazione / "src").is_dir():
            print("Clonato in %s" % destinazione)
            return destinazione
        print("Clone non riuscito: %s" % (risultato.stderr.strip().splitlines() or [""])[-1])

    if BUNDLE_URL:
        print("Scarico il bundle da %s ..." % BUNDLE_URL)
        from urllib.request import urlretrieve

        zip_path = Path(tempfile.gettempdir()) / "pianovam_video2midi_bundle.zip"
        urlretrieve(BUNDLE_URL, zip_path)
    else:
        try:
            from google.colab import files
        except ImportError:
            raise RuntimeError(
                "Non trovo il codice in %s e non posso chiedertelo (non sei in Colab).\n"
                "Clona il repo a mano:  git clone %s %s" % (destinazione, GITHUB_REPO, destinazione))
        print("Carica lo zip del progetto (src/ + pesi, ~1,5 MB).")
        print("Non ce l'hai? Sul tuo computer:  python3 scripts/pack_colab_bundle.py")
        caricati = files.upload()
        if not caricati:
            raise RuntimeError("nessun file caricato")
        zip_path = Path(next(iter(caricati)))

    destinazione.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archivio:
        archivio.extractall(destinazione)
    print("Estratto in %s" % destinazione)
    return destinazione

In [ ]:
# ======================================================================
# 1) IL MODELLO: codice, pesi, dipendenze
# ======================================================================
titolo("1) IL MODELLO")
code_dir = scarica_o_chiedi_modello(CODE_DIR)

# Le dipendenze che servono davvero qui: av (decodifica video), cv2 (rettifica
# prospettica della tastiera), pretty_midi (scrittura del MIDI). numpy, torch,
# pandas e matplotlib in Colab ci sono gia'.
for modulo, pacchetto in (("av", "av"), ("cv2", "opencv-python-headless"),
                          ("pretty_midi", "pretty_midi"), ("torch", "torch")):
    try:
        importlib.import_module(modulo)
    except ImportError:
        print("Installo %s ..." % pacchetto)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pacchetto], check=False)

if str(code_dir) not in sys.path:
    sys.path.insert(0, str(code_dir))

import torch                                                             # noqa: E402
from src import decode, geometry, infer, keys, manifest                  # noqa: E402
from src import metrics, model as model_mod, targets, video               # noqa: E402

if DEVICE == "auto":
    device = "cuda" if torch.cuda.is_available() else (
        "mps" if torch.backends.mps.is_available() else "cpu")
else:
    device = DEVICE

checkpoint = Path(CHECKPOINT) if CHECKPOINT else code_dir / "runs/m6_2ch/last.pt"
if not checkpoint.is_file():
    disponibili = sorted((code_dir / "runs").glob("*/*.pt"))
    raise FileNotFoundError(
        "pesi non trovati: %s\nDisponibili: %s" % (checkpoint, [str(p) for p in disponibili]))

model = model_mod.load_checkpoint(checkpoint, device)
raggio = model_mod.temporal_radius_for_checkpoint(checkpoint)
print("Pesi            : %s (%s)" % (checkpoint, human(checkpoint.stat().st_size)))
print("Modello         : %d parametri, %d canali in ingresso, raggio temporale %d"
      % (model_mod.count_parameters(model), model_mod.in_channels_from_checkpoint(checkpoint), raggio))
print("Dispositivo     : %s" % device)
print("Soglia sugli onset: %.2f (modificabile nella cella 0)" % THRESHOLD)
print()
print("Come legge il video: ritaglia la tastiera, la raddrizza in una striscia di 96x1056")
print("pixel (88 colonne da 12 px, una per tasto) e per ogni frame guarda anche le")
print("differenze con i frame vicini (cosi' il *movimento* dei tasti e' un segnale esplicito).")
print("Da li' stima quattro mappe di probabilita' per tasto: onset, tasto premuto, rilascio,")
print("velocity. Le note si leggono dai picchi della mappa di onset.")

In [ ]:
# ======================================================================
# 2) L'ESEMPIO: indice 1..107 -> una registrazione
# ======================================================================
titolo("2) L'ESEMPIO")
if MODE != "dataset":
    print("MODE = %r: salto la scelta dell'esempio. Il video tuo lo carichi nella cella 7." % MODE)
else:
    try:
        dataset_root = find_dataset_root(DATASET_ROOT)
    except FileNotFoundError as errore:
        dataset_root = Path(DATASET_ROOT).resolve()
        dataset_root.mkdir(parents=True, exist_ok=True)
        print(errore)
        print()
        print("Provo a scaricare metadata.json (66 KB) in %s" % dataset_root)
        if file_dataset(dataset_root, "metadata.json") is None:
            raise
    meta_all = json.loads((dataset_root / "metadata.json").read_text(encoding="utf-8"))
    ordine = sorted(meta_all, key=int)

    # La tabella: un indice da 1 a 107, cosi' scegliere e' solo "RECORD_INDEX = 42".
    righe = []
    for posizione, chiave in enumerate(ordine, start=1):
        voce = meta_all[chiave]
        tempo = voce.get("record_time", chiave)
        righe.append({"indice": posizione,
                      "record_time": tempo,
                      "pianista": voce.get("P1_name", "?"),
                      "split": voce.get("split", "?"),
                      "brano": str(voce.get("piece", ""))[:28],
                      "durata": durata_breve(voce.get("duration", "?")),
                      "video": "sul disco" if (dataset_root / "Video" / (tempo + ".mp4")).is_file()
                               else "~324 MB dal Hub"})
    tabella = pd.DataFrame(righe)
    print("Il dataset ha %d registrazioni. Scegli con RECORD_INDEX (1..%d) in cima al notebook,"
          % (len(ordine), len(ordine)))
    print("oppure con RECORD_TIME (il nome del file). Qui sotto la tabella completa:")
    print()
    mostra_tabella(tabella)

    # Chi elaboriamo.
    per_tempo = {voce.get("record_time", chiave): chiave for chiave, voce in meta_all.items()}
    if RECORD_TIME:
        chiave = per_tempo.get(RECORD_TIME)
        if chiave is None:
            raise SystemExit("RECORD_TIME = %r non esiste: guarda la tabella qui sopra." % RECORD_TIME)
    elif RECORD_INDEX:
        if not 1 <= int(RECORD_INDEX) <= len(ordine):
            raise SystemExit("RECORD_INDEX deve stare fra 1 e %d" % len(ordine))
        chiave = ordine[int(RECORD_INDEX) - 1]
    else:
        raise SystemExit("nella cella 0 metti RECORD_INDEX (1..%d) oppure RECORD_TIME." % len(ordine))

    record = meta_all[chiave]
    record_time = record.get("record_time", chiave)
    indice_scelto = ordine.index(chiave) + 1

    print()
    print("=" * 78)
    print("Esempio %d di %d: %s" % (indice_scelto, len(ordine), record_time))
    print("=" * 78)
    print("brano            : %s, %s" % (record.get("composer", "?"), record.get("piece", "?")))
    print("pianista         : %s (%s, %s, skill %s, music major %s)"
          % (record.get("P1_name", "?"), record.get("P1_gender", "?"), record.get("P1_age", "?"),
             record.get("P1_skill", "?"), record.get("P1_musicmajor", "?")))
    if record.get("P2_name"):
        print("secondo pianista : %s" % record["P2_name"])
    print("split ufficiale  : %s" % record.get("split", "?"))
    print("durata           : %s" % durata_breve(record.get("duration", "?")))
    print("registrato       : %s" % record.get("record_time", "?"))
    print("calibrazione     : LT %s | RT %s | RB %s | LB %s"
          % (record.get("Point_LT", "?"), record.get("Point_RT", "?"),
             record.get("Point_RB", "?"), record.get("Point_LB", "?")))
    if str(record.get("split")) == "test":
        print("NB: questa registrazione sta nel *test ufficiale*: il modello non l'ha mai vista.")
    for cartella, estensione in (("Video", "mp4"), ("MIDI", "mid"), ("TSV", "tsv")):
        percorso = dataset_root / cartella / ("%s.%s" % (record_time, estensione))
        stato = human(percorso.stat().st_size) if percorso.is_file() else "da scaricare dal Hub"
        print("%-16s : %s" % (cartella, stato))
    print()
    print("Per cambiare esempio: RECORD_INDEX nella cella 0 e riesegui da qui in giu'.")

In [ ]:
# ======================================================================
# 3) VIDEO + AUDIO ORIGINALE
# ======================================================================
# Il video di una registrazione pesa ~340 MB: lo scarichiamo una volta sul disco e
# nell'output del notebook mettiamo solo un estratto dei primi VIDEO_EXCERPT_SECONDS
# secondi a 640 px. L'audio di quell'estratto e' quello registrato dal pianoforte.
titolo("3) VIDEO E AUDIO ORIGINALE")
if MODE != "dataset":
    print("MODE = %r: il video tuo lo carichi nella cella 7." % MODE)
else:
    video_path = file_dataset(dataset_root, "Video/%s.mp4" % record_time)
    if video_path is None:
        print("Il video non e' disponibile: le celle successive si fermano qui.")
    else:
        print("Video/%s.mp4 -> %s" % (record_time, human(video_path.stat().st_size)))
        try:
            comando = ["ffprobe", "-v", "error", "-print_format", "json",
                       "-show_streams", "-show_format", str(video_path)]
            info = json.loads(subprocess.run(comando, check=True, capture_output=True, text=True).stdout)
            for stream in info["streams"]:
                if stream["codec_type"] == "video":
                    num, den = stream["avg_frame_rate"].split("/")
                    print("Video : %s %sx%s, %.2f fps, %s frame"
                          % (stream["codec_name"], stream["width"], stream["height"],
                             float(num) / float(den), stream.get("nb_frames", "?")))
                else:
                    print("Audio : %s, %s Hz, %s canali"
                          % (stream["codec_name"], stream.get("sample_rate"), stream.get("channels")))
            print("Durata: %.2f s" % float(info["format"]["duration"]))
        except Exception as errore:
            print("ffprobe non disponibile (%s). In Colab c'e'." % errore)
        if "jiwoo" in str(record.get("P1_name", "")).lower():
            print("NB: in questa registrazione il busto della pianista e' sfocato per privacy;")
            print("    tastiera e mani sono visibili.")
        if SHOW_VIDEO:
            secondi_estratto = VIDEO_EXCERPT_SECONDS
            if SECONDS is not None:
                secondi_estratto = min(secondi_estratto, int(SECONDS))
            estratto = Path(tempfile.gettempdir()) / (record_time + "_video_estratto.mp4")
            comando = ["ffmpeg", "-y", "-loglevel", "error", "-ss", "0", "-t", str(secondi_estratto),
                       "-i", str(video_path), "-vf", "scale=640:-2", "-c:v", "libx264",
                       "-preset", "veryfast", "-crf", "30", "-c:a", "aac", "-b:a", "64k", str(estratto)]
            try:
                from IPython.display import Video, display

                subprocess.run(comando, check=True)
                print()
                print("Estratto dei primi %d s, a 640 px (%s): l'audio e' quello originale."
                      % (secondi_estratto, human(estratto.stat().st_size)))
                display(Video(str(estratto), embed=True, width=640))
                print("File completo sul disco: %s" % video_path)
            except Exception as errore:
                print("Anteprima non creata (%s). Servono ffmpeg e IPython (in Colab ci sono)." % errore)
                print("Il file completo lo apri a mano: %s" % video_path)
        else:
            print("(Anteprima video non mostrata: SHOW_VIDEO = False.)")

In [ ]:
# ======================================================================
# 4) IL MIDI ORIGINALE (quello registrato dal pianoforte digitale)
# ======================================================================
# Il sintetizzatore qui sotto serve due volte: per far sentire il MIDI originale e
# per far sentire quello stimato. E' lo *stesso* suono, quindi nell'ascolto A/B
# l'unica differenza che senti sono le note: quelle che il modello indovina e
# quelle che si perde. (Un pianoforte vero lo senti nel video della cella 3.)
MAX_RING = 1.5    # secondi: oltre questo il suono e' gia' finito e i pedali durerebbero troppo


def sintetizza(elenco_note):
    """Note (midi, inizio_s, fine_s, velocity) -> audio mono float32 in [-1, 1]."""
    if not elenco_note:
        return np.zeros(0, dtype=np.float32)
    ultimo = max(nota[2] for nota in elenco_note)
    campioni = np.zeros(int((ultimo + MAX_RING + 0.2) * SAMPLE_RATE), dtype=np.float32)
    for midi, inizio, fine, velocity in elenco_note:
        frequenza = 440.0 * 2.0 ** ((midi - 69) / 12.0)
        lunghezza = int(min(max(fine - inizio, 0.05), MAX_RING) * SAMPLE_RATE)
        tempo = np.arange(lunghezza) / SAMPLE_RATE
        # attacco rapidissimo e decadimento: il timbro di un tasto, non di un violino
        inviluppo = np.exp(-tempo / 0.30) * (1.0 - np.exp(-tempo / 0.004))
        coda = min(20, lunghezza)
        inviluppo[-coda:] *= np.linspace(1.0, 0.0, coda)
        suono = (np.sin(2 * np.pi * frequenza * tempo)
                 + 0.45 * np.sin(2 * np.pi * 2 * frequenza * tempo)
                 + 0.18 * np.sin(2 * np.pi * 3 * frequenza * tempo))
        avvio = int(inizio * SAMPLE_RATE)
        campioni[avvio:avvio + lunghezza] += (velocity / 127.0) * inviluppo * suono
    picco = float(np.abs(campioni).max())
    return campioni / picco * 0.85 if picco > 0 else campioni


note_originali = []          # (midi, inizio_s, fine_s, velocity)
midi_originale = None
titolo("4) IL MIDI ORIGINALE")
if MODE != "dataset":
    print("MODE = %r: nessun MIDI di riferimento, si va diretti alla stima (cella 5)." % MODE)
else:
    midi_path = file_dataset(dataset_root, "MIDI/%s.mid" % record_time)
    if midi_path is None:
        print("Il MIDI di riferimento non e' disponibile.")
    else:
        import pretty_midi

        midi_originale = pretty_midi.PrettyMIDI(str(midi_path))
        programmi = sorted({int(strumento.program) for strumento in midi_originale.instruments})
        for strumento in midi_originale.instruments:
            for nota in strumento.notes:
                if not strumento.is_drum:
                    note_originali.append((nota.pitch, nota.start, nota.end, nota.velocity))
        note_originali.sort(key=lambda nota: (nota[1], nota[0]))
        print("MIDI/%s.mid -> %s" % (record_time, human(midi_path.stat().st_size)))
        print("Tracce          : %d, programmi %s" % (len(midi_originale.instruments), programmi))
        print("Tempo iniziale  : %.2f BPM (stimato dal file), durata %.2f s"
              % (midi_originale.estimate_tempo(), midi_originale.get_end_time()))
        print("Note            : %d" % len(note_originali))
        if note_originali:
            altezze = [nota[0] for nota in note_originali]
            print("Ambito          : %s - %s (MIDI %d - %d)"
                  % (keys.note_name(min(altezze)), keys.note_name(max(altezze)),
                     min(altezze), max(altezze)))
            print()
            print("Prime 5 note (durate calcolate dalla fine della nota):")
            for midi, inizio, fine, velocity in note_originali[:5]:
                print("  t = %7.3f s  %-4s MIDI %3d  durata %6.0f ms  velocity %3d"
                      % (inizio, keys.note_name(midi), midi, (fine - inizio) * 1000.0, velocity))
        # Per l'ascolto teniamo la stessa finestra che analizzera' il modello.
        finestra = None if SECONDS is None else float(SECONDS)
        ascolto = [nota for nota in note_originali if finestra is None or nota[1] < finestra]
        print()
        print("Ascolto delle prime %s s (%d note) col sintetizzatore condiviso:"
              % ("%.0f" % finestra if finestra else "tutte", len(ascolto)))
        riproduci_audio(sintetizza(ascolto), SAMPLE_RATE, "%s_originale" % record_time)
        print("File MIDI originale: %s" % midi_path)
        # Per scaricarlo su Colab basta togliere il commento:
        #   from google.colab import files; files.download(str(midi_path))

In [ ]:
# ======================================================================
# 5) LA STIMA: video -> tastiera rettificata -> note -> MIDI
# ======================================================================
# Tre passi: (a) ritaglio la tastiera e la raddrizzo in una striscia di 88 colonne,
# salvandola in una cache (si fa una volta sola, poi e' veloce); (b) il modello
# stima per ogni tasto e ogni frame quattro mappe di probabilita'; (c) dalle
# probabilita' leggo le note con i picchi e la soglia THRESHOLD.
titolo("5) LA STIMA")
out_dir = Path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
frames_max = None if SECONDS is None else int(round(float(SECONDS) * decode.FPS))


def parse_quad(testo):
    """'x,y x,y x,y x,y' -> (4, 2) float32, nell'ordine LT RT RB LB (come il dataset).

    I numeri si estraggono con una regex, cosi' va bene anche il formato di
    metadata.json ("91, 337" con lo spazio dopo la virgola) e le virgole in piu'.
    """
    numeri = re.findall(r"-?[0-9]+(?:\\.[0-9]+)?", testo)
    if len(numeri) != 8:
        raise ValueError("servono 8 numeri (4 punti x, y), trovati %d in %r" % (len(numeri), testo))
    return np.array(numeri, dtype=np.float32).reshape(4, 2)


def sorgente_e_quad():
    """(file video, rettangolo della tastiera, nome): dal dataset o dal video caricato."""
    if MODE == "dataset":
        sorgente = file_dataset(dataset_root, "Video/%s.mp4" % record_time)
        if sorgente is None:
            raise RuntimeError("il video di questa registrazione non e' disponibile")
        quad = manifest.quad_for_record(dataset_root, record_time, top_margin=TOP_MARGIN)
        return sorgente, quad, record_time
    sorgente = Path(UPLOAD_VIDEO) if UPLOAD_VIDEO else out_dir / "video_caricato.mp4"
    if not sorgente.is_file():
        raise RuntimeError("nessun video da analizzare: caricalo nella cella 7")
    if not QUAD:
        raise RuntimeError("manca QUAD: leggi i 4 angoli della tastiera nella cella 7")
    punti = parse_quad(QUAD)
    angoli = {corner: "%s,%s" % (punto[0], punto[1])
              for corner, punto in zip(geometry.CORNER_ORDER, punti)}
    return sorgente, geometry.quad_from_metadata(angoli, top_margin=TOP_MARGIN), sorgente.stem


def cache_di_strisce(nome, sorgente, quad):
    """Cache delle strisce rettificate: la riuso se c'e' gia', altrimenti la costruisco."""
    candidati = []
    if CACHE_ROOT:
        candidati.append(Path(CACHE_ROOT) / nome)
    candidati.append(out_dir / "cache" / nome)
    if MODE == "dataset":
        candidati.append(Path(dataset_root) / "cache" / nome)
    candidati.append(Path(code_dir) / "data/cache" / nome)
    for candidato in candidati:
        if (candidato / "meta.json").is_file():
            print("Cache delle strisce gia' pronta: %s" % candidato)
            return candidato, False
    destinazione = out_dir / "cache" / nome
    print("Ritaglio e raddrizzo la tastiera di %s ..." % sorgente.name)
    iniziato = datetime.datetime.now()
    meta = video.build_cache(sorgente, quad, destinazione, max_frames=frames_max)
    peso = sum(p.stat().st_size for p in video.chunk_files(destinazione))
    print("  %d frame in %.1f s -> %s (la striscia di una registrazione intera sta in ~30 MB)"
          % (meta["n_frames"], (datetime.datetime.now() - iniziato).total_seconds(), human(peso)))
    return destinazione, True


def analizza(sorgente, quad, nome):
    """Il giro completo su un video: cache, modello, note, MIDI. Restituisce le note."""
    cache_dir, costruita = cache_di_strisce(nome, sorgente, quad)
    frames_cache = int(video.read_meta(cache_dir)["n_frames"])
    print("Strisce : %d frame (%.1f s) in %s" % (frames_cache, frames_cache / decode.FPS, cache_dir))
    if not costruita and frames_max is not None and frames_cache > frames_max:
        print("Questa cache e' piu' lunga di SECONDS = %s: il modello la analizza tutta e poi" % SECONDS)
        print("tengo solo i primi %s s per il confronto e per il MIDI." % SECONDS)

    # Orientamento: se la tastiera fosse ripresa "al contrario" la raddrizzo. E' un test
    # geometrico sui tasti neri: costa niente e non c'entra con l'addestramento.
    indici = np.linspace(0, frames_cache - 1, num=min(16, frames_cache)).astype(int)
    campioni = np.stack([video.read_strips(cache_dir, int(i), 1)[0] for i in indici])
    input_flip = keys.detect_input_flip(campioni)
    normale = float(np.mean([keys.black_key_score(f) for f in campioni]))
    specchiata = float(np.mean([keys.black_key_score(f[:, ::-1]) for f in campioni]))
    print("Tasti neri: %.2f come nel dataset, %.2f specchiata -> %s"
          % (normale, specchiata, "la raddrizzo io" if input_flip else "va bene cosi'"))

    iniziato = datetime.datetime.now()
    probabilita = infer.predict_recording(model, cache_dir, device=device, window=WINDOW, stride=STRIDE,
                                          tta_flip=True, input_flip=input_flip, temporal_radius=raggio)
    trovate = decode.notes_from_heads(probabilita, threshold_onset=THRESHOLD, min_distance=2)
    frames_usati = min(frames_cache, frames_max) if frames_max else frames_cache
    trovate = [nota for nota in trovate if nota.start_frame < frames_usati]
    percorso = out_dir / ("%s_stimato.mid" % nome)
    decode.notes_to_midi(trovate, percorso)
    trascorso = (datetime.datetime.now() - iniziato).total_seconds()

    print()
    print("Modello : %.0f s per %d frame (%.2f s di video), soglia %.2f"
          % (trascorso, frames_usati, frames_usati / decode.FPS, THRESHOLD))
    print("Note    : %d stimate, da t = 0 a %.1f s" % (len(trovate), frames_usati / decode.FPS))
    if trovate:
        durate = np.array([nota.end - nota.start for nota in trovate])
        print("          durata mediana %.0f ms, la piu' lunga %.2f s"
              % (1000.0 * float(np.median(durate)), float(durate.max())))
    if note_originali:
        vere = [nota for nota in note_originali if nota[1] < frames_usati / decode.FPS]
        print("          nel MIDI originale, nella stessa finestra, ce ne sono %d" % len(vere))
    print()
    print("MIDI stimato: %s" % percorso)
    # Per scaricarlo su Colab:  from google.colab import files; files.download(str(percorso))
    return trovate, frames_usati, percorso


sorgente, quad, nome = sorgente_e_quad()
note_stimate, frames_analizzati, percorso_stimato = analizza(sorgente, quad, nome)


In [ ]:
# ======================================================================
# 6) IL CONFRONTO: metriche, piano roll, ascolto A/B
# ======================================================================
# La tolleranza e' di 3 frame = 50 ms: a 60 fps un onset e' un istante, e sotto
# i 50 ms due note sono la stessa nota.
titolo("6) IL CONFRONTO")
secondi = frames_analizzati / decode.FPS
stimate_per_ascolto = [(keys.index_to_note(nota.key), nota.start, nota.end, nota.velocity)
                       for nota in note_stimate]
vere_per_ascolto = [nota for nota in note_originali if nota[1] < secondi]

if MODE == "dataset" and note_originali:
    tsv_path = file_dataset(dataset_root, "TSV/%s.tsv" % record_time)
    if tsv_path is not None:
        eventi_veri = targets.read_tsv(tsv_path).subset(0, frames_analizzati)
        conteggi = metrics.note_metrics(eventi_veri, note_stimate, tolerance=metrics.DEFAULT_TOLERANCE)
        tolleranza_ms = 1000.0 * metrics.DEFAULT_TOLERANCE / decode.FPS
        print("Verita' di riferimento: TSV/%s.tsv (note registrate dal pianoforte digitale)." % record_time)
        print("Confronto sulle prime %.1f s, tolleranza %.0f ms:" % (secondi, tolleranza_ms))
        print("  note vere    : %d" % conteggi["n_gt"])
        print("  note stimate : %d" % conteggi["n_pred"])
        print("  azzeccate    : %d" % conteggi["matched"])
        print("  precision    : %.3f   delle note che il modello scrive, quante esistono davvero"
              % conteggi["precision"])
        print("  recall       : %.3f   delle note che esistono, quante il modello ne scrive"
              % conteggi["recall"])
        print("  F1           : %.3f" % conteggi["f1"])
        print("  errore medio sugli onset: %.1f ms" % conteggi["mean_onset_error_ms"])
        print()
        print("Come si legge: su questo modello l'errore tipico e' 'non vedere' le note")
        print("(recall piu' basso della precision), non inventarle. Se il recall e' molto piu'")
        print("basso, il caso peggiore e' la musica densa e veloce: li' abbassare THRESHOLD")
        print("recupera qualcosa ma riempie anche di note che non ci sono.")
elif MODE == "upload":
    print("Video tuo: nessun MIDI di riferimento, quindi niente metriche.")
    print("Quello che puoi valutare a orecchio e a occhio: ascolto A/B e piano roll qui sotto.")
else:
    print("MIDI di riferimento non disponibile: niente metriche.")
print()
print("Ascolto A/B (stesso sintetizzatore, cosi' senti solo le differenze di note):")
if vere_per_ascolto:
    print("  MIDI originale: %d note" % len(vere_per_ascolto))
    riproduci_audio(sintetizza(vere_per_ascolto), SAMPLE_RATE, "%s_originale_ascolto" % nome)
else:
    print("  MIDI originale: non c'e' (video tuo, nessun riferimento da confrontare)")
print("  MIDI stimato  : %d note" % len(stimate_per_ascolto))
riproduci_audio(sintetizza(stimate_per_ascolto), SAMPLE_RATE, "%s_stimato_ascolto" % nome)

In [ ]:
# ======================================================================
# 6b) PIANO ROLL: il MIDI originale e quello stimato, uno sopra l'altro
# ======================================================================
# Stessa scala di tempo e di altezza: le note che mancano si vedono come righe
# presenti sopra e assenti sotto, quelle inventate il contrario.
titolo("6b) PIANO ROLL")
import matplotlib                                # noqa: E402

matplotlib.use("Agg")
import matplotlib.pyplot as plt                  # noqa: E402


def piano_roll(elenco_vero, elenco_stimato, secondi, percorso):
    """Due pannelli (originale, stimato) con lo stesso asse di tempo e di altezza."""
    figura, assi = plt.subplots(2, 1, figsize=(12, 5.5), sharex=True, sharey=True)
    pannelli = ((assi[0], elenco_vero, "MIDI originale", "#1f77b4"),
                (assi[1], elenco_stimato, "MIDI stimato", "#d62728"))
    for asse, elenco, etichetta, colore in pannelli:
        for midi, inizio, fine, _velocity in elenco:
            asse.plot([inizio, fine], [midi, midi], color=colore, linewidth=2.2,
                      solid_capstyle="butt")
        asse.set_ylabel(etichetta, fontsize=10)
        asse.grid(alpha=0.22, linewidth=0.5)
        asse.set_title("%s: %d note" % (etichetta, len(elenco)), loc="left", fontsize=10)
    assi[1].set_xlabel("secondi dall'inizio del video")
    assi[0].set_xlim(0, max(float(secondi), 1.0))
    assi[0].set_ylim(20, 109)
    altezze = list(range(24, 109, 12))
    assi[0].set_yticks(altezze)
    assi[0].set_yticklabels([keys.note_name(q) for q in altezze], fontsize=9)
    figura.suptitle("%s: prime %.0f s di registrazione, soglia %.2f" % (nome, secondi, THRESHOLD),
                    fontsize=11)
    figura.tight_layout()
    figura.savefig(percorso, dpi=110)
    plt.close(figura)
    return percorso


percorso_figura = piano_roll(vere_per_ascolto, stimate_per_ascolto, secondi,
                             out_dir / ("%s_piano_roll.png" % nome))
print("Figura: %s" % percorso_figura)
try:
    from IPython.display import Image, display

    display(Image(filename=str(percorso_figura)))
except Exception:
    print("(fuori da Colab la figura non si vede qui: aprila dal percorso qui sopra)")
print()
print("File da portarti via:")
print("  MIDI stimato  : %s" % percorso_stimato)
if midi_originale is not None:
    print("  MIDI originale: %s" % midi_path)
print("  piano roll    : %s" % percorso_figura)
print()
print("Su Colab, per scaricarli, togli il commento:")
print("  from google.colab import files; files.download(str(percorso_stimato))")

## Video tuo

Questa parte serve solo con `MODE = "upload"` nella cella 0. Il modello funziona anche su
un video caricato a mano: l'unica cosa che il dataset ti dava gratis e' la **calibrazione
della tastiera**, cioe' i 4 angoli del rettangolo della tastiera nel primo frame. Per un
video tuo li devi indicare tu, leggendoli sulla griglia che questa cella disegna sopra il
primo frame (griglia ogni 100 pixel).

Come si scrivono, nell'ordine **LT RT RB LB** (sinistra-alto, destra-alto, destra-basso,
sinistra-basso):

```python
QUAD = "121,355 1839,345 1839,558 120,564"
```

Due note oneste:

- il **rilevamento automatico** della tastiera e' ancora in sviluppo: oggi il notebook non
  te lo propone, ti chiede i 4 angoli e basta. Se sbagli di qualche pixel il modello
  funziona lo stesso (la striscia viene ricampionata), se sbagli di molto il risultato crolla;
- il **formato del video** conta piu' della qualita': la tastiera intera in campo, vista
  dall'alto o di poco inclinata, 60 fps. Con una inquadratura diversa da quelle del dataset
  aspettati un risultato molto piu' debole (il modello non ha mai visto altro).

In [ ]:
# ======================================================================
# 7) VIDEO TUO: carica, leggi gli angoli sulla griglia, stima
# ======================================================================
titolo("7) VIDEO TUO")
if MODE != "upload":
    print("Questa cella serve con MODE = \"upload\" nella cella 0.")
    print("Con MODE = \"dataset\" hai gia' tutto: video, MIDI originale e confronto (celle 3-6).")
else:
    out_dir = Path(OUT_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)
    frames_max = None if SECONDS is None else int(round(float(SECONDS) * decode.FPS))

    if UPLOAD_VIDEO:
        sorgente = Path(UPLOAD_VIDEO)
        print("Uso il video indicato in UPLOAD_VIDEO: %s" % sorgente)
    else:
        from google.colab import files

        print("Carica il tuo video (mp4). Va benissimo anche un estratto di 30 s:")
        print("il modello analizza comunque i primi %s secondi (SECONDS)." % SECONDS)
        caricati = files.upload()
        if not caricati:
            raise RuntimeError("nessun file caricato")
        nome_file = next(iter(caricati))
        sorgente = out_dir / nome_file
        sorgente.write_bytes(caricati[nome_file])
        print("Salvato in %s (%s)" % (sorgente, human(sorgente.stat().st_size)))

    # La griglia per leggere gli angoli della tastiera sul primo frame.
    import av
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    with av.open(str(sorgente)) as contenitore:
        primo_frame = next(contenitore.decode(video=0)).to_ndarray(format="rgb24")
    altezza, larghezza = primo_frame.shape[:2]
    print("Frame: %d x %d pixel" % (larghezza, altezza))

    figura, asse = plt.subplots(figsize=(14, 14.0 * altezza / larghezza))
    asse.imshow(primo_frame)
    asse.set_xticks(range(0, larghezza, 100))
    asse.set_yticks(range(0, altezza, 100))
    asse.grid(color="yellow", alpha=0.35, linewidth=0.5)
    asse.tick_params(labelsize=7)
    figura.tight_layout()
    percorso_griglia = out_dir / ("%s_griglia.png" % sorgente.stem)
    figura.savefig(percorso_griglia, dpi=90)
    plt.close(figura)
    print("Griglia salvata in %s (ogni quadretto = 100 px)" % percorso_griglia)
    try:
        from IPython.display import Image, display

        display(Image(filename=str(percorso_griglia)))
    except Exception:
        print("(fuori da Colab apri la figura dal percorso qui sopra)")

    print()
    print("Sulla tastiera leggi i 4 angoli, nell'ordine LT RT RB LB, e scrivili nella cella 0:")
    print('  QUAD = "121,355 1839,345 1839,558 120,564"')
    if QUAD is None:
        print()
        print("QUAD e' ancora None: mettilo nella cella 0 e riesegui questa cella.")
    else:
        UPLOAD_VIDEO = str(sorgente)      # cosi' il resto del notebook usa questo file
        _, quad_upload, nome_upload = sorgente_e_quad()
        note_stimate, frames_analizzati, percorso_stimato = analizza(sorgente, quad_upload, nome_upload)
        secondi = frames_analizzati / decode.FPS
        stimate_per_ascolto = [(keys.index_to_note(nota.key), nota.start, nota.end, nota.velocity)
                               for nota in note_stimate]
        vere_per_ascolto = []
        nome = nome_upload
        print()
        print("Ascolto del MIDI stimato (%d note su %.1f s):" % (len(stimate_per_ascolto), secondi))
        riproduci_audio(sintetizza(stimate_per_ascolto), SAMPLE_RATE, "%s_stimato_ascolto" % nome)
        percorso_figura = piano_roll(vere_per_ascolto, stimate_per_ascolto, secondi,
                                     out_dir / ("%s_piano_roll.png" % nome))
        print("Piano roll (senza il pannello originale: il riferimento non c'e'): %s" % percorso_figura)
        try:
            from IPython.display import Image, display

            display(Image(filename=str(percorso_figura)))
        except Exception:
            pass
        print()
        print("Per scaricare il MIDI stimato:")
        print("  from google.colab import files; files.download(str(percorso_stimato))")

## Cosa c'e' dietro, e cosa non funziona ancora

**Il modello.** La tastiera viene ritagliata e raddrizzata in una striscia di 88 colonne (una
per tasto, 12 px l'una): al modello non arriva mai il video, solo quella striscia piu' le
differenze con i frame vicini, cosi' il *movimento* dei tasti e' un segnale esplicito invece
di qualcosa da dedurre. Da li' quattro teste (onset, tasto premuto, rilascio, velocity) e le
note lette dai picchi della mappa di onset. Poco piu' di 0,3 milioni di parametri: sta in
una manciata di MB e gira su CPU.

**I limiti, in chiaro:**

- **soglia fissa.** `THRESHOLD = 0.70` e' stata scelta sui pianisti **visti** in
  addestramento. Su un pianista mai visto la probabilita' di onset e' sistematicamente piu'
  bassa, quindi la stessa soglia taglia troppo: se il recall e' molto piu' basso della
  precision, prova `THRESHOLD = 0.4` o `0.5` nella cella 0. Non e' una correzione magica:
  su musica densa recupera recall e peggiora precision (misurato: da 0.30 di soglia su
  Minwook +0.09 di F1, ma ancora precision 0.55 / recall 0.41);
- **checkpoint.** I pesi sono `runs/m6_2ch/last.pt`, cioe' l'ultima epoca (9) di
  addestramento: e' stata scelta guardando l'F1 sulle **note** nella validazione ufficiale
  (pianisti visti), non la loss sui pixel, perche' la loss si satura molto prima dell'F1;
- **rilevamento automatico della tastiera**: non c'e'. Per un video tuo i 4 angoli li
  indichi a mano (cella 7). Questo e' il pezzo di lavoro aperto piu' grosso;
- **registrazioni a quattro mani**: due pianisti nello stesso video, fuori dalla
  distribuzione di addestramento. Se scegli una di quelle, aspettati poco;
- **velocity**: la forza del tasto si legge male dal video (l'etichetta del dataset e' il
  MIDI del pianoforte digitale, il segnale visivo e' debole). L'onset e' affidabile, la
  velocity e' grossolana.

**Licenza e citazione.** Dataset CC BY-NC-SA 4.0 (solo uso non commerciale):

```bibtex
@inproceedings{kim2025pianovam,
  title={PianoVAM: A Multimodal Piano Performance Dataset},
  author={Kim, Yonghyun and Park, Junhyung and Bae, Joonhyung and Kim, Kirak and Kwon, Taegyun and Lerch, Alexander and Nam, Juhan},
  booktitle={Proceedings of the 26th International Society for Music Information Retrieval Conference (ISMIR)},
  year={2025}
}
```